# Módulo 4 — Modelos profundos de una clase y detección secuencial

Tres modelos, cada uno nacido de una limitación concreta que dejó a la vista el benchmark del Módulo 3:

| Modelo | Limitación que ataca | Unidad evaluada |
|---|---|---|
| **VAE** | El mejor detector fue un modelo de densidad (GMM); el autoencoder solo reconstruye, no modela densidad | Transacción |
| **Deep SVDD** | El autoencoder apenas superó su ablación lineal (0.581 vs 0.487) — ¿el problema es la profundidad o el *objetivo*? | Transacción |
| **Autoencoder GRU** | Los trece detectores tratan cada fila como independiente; una cuenta mula es anómala por su *historia* | **Cuenta destino** |

Los dos primeros comparten split, escalado y métricas con el Módulo 3, así que sus números entran en la misma tabla. El tercero cambia la unidad de análisis y se reporta aparte: comparar un PR-AUC por cuenta con uno por transacción sería comparar dos problemas distintos.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.deep.one_class import DeepSVDDDetector, VAEDetector
from src.deep.sequences import (
    FEATURE_NAMES,
    aggregate_step_errors,
    fraud_window_coverage,
    get_sequence_data,
    sequence_step_errors,
    train_sequence_autoencoder,
)
from src.data.loader import load_raw_data
from src.deep.train_deep import plot_deep_pr_curves, plot_sequence_detector, summary_table
from src.unsupervised.loader import get_unsupervised_data
from src.unsupervised.models import anomaly_score
from src.unsupervised.train_unsupervised import evaluate

## 1. Nivel transacción: VAE y Deep SVDD

Mismo split que los Módulos 2 y 3 (`get_unsupervised_data`, semilla fija) y el mismo `RobustScaler`: sin eso, cualquier diferencia de métrica podría venir de los datos y no del modelo.

In [ ]:
X_train, X_test, y_test = get_unsupervised_data()

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train (solo normales): {X_train.shape}")
print(f"Test (mixto): {X_test.shape} — fraude: {int(y_test.sum())} ({y_test.mean():.2%})")
print(f"Rango tras escalar: |x|max = {np.abs(X_train_scaled).max():.0f}")

### Una nota de estabilidad numérica que no es opcional

Ese `|x|max` de arriba es el detalle que decide si el VAE entrena o produce `NaN`. Tras el `RobustScaler` las colas de `errorBalanceDest` llegan a ~1900, y la suma de cuadrados por fila alcanza 3.7e6. Con el término de reconstrucción **sumado** sobre las 15 columnas, la pérdida arranca en millones, los gradientes explotan y los pesos se vuelven `NaN` en la primera época.

Tres decisiones lo evitan, todas en `src/deep/one_class.py`:

- el error de reconstrucción se promedia sobre las features en vez de sumarse — lo deja en la misma escala que el autoencoder del Módulo 2, que sí converge sobre estos datos;
- la log-varianza se acota a [-10, 10], para que `exp(logvar)` no se desborde;
- se recorta la norma del gradiente a 5, porque las colas producen lotes con pérdidas de órdenes de magnitud mayores que el resto.

In [ ]:
resultados = {}

for nombre, detector in [("vae", VAEDetector()), ("deep_svdd", DeepSVDDDetector())]:
    detector.fit(X_train_scaled)
    scores = anomaly_score(detector, X_test_scaled)
    metricas = evaluate(y_test, scores)
    resultados[nombre] = {"scores": scores, **metricas}
    print(f"{nombre}: PR-AUC={metricas['pr_auc']:.4f}")

### Deep SVDD: verificar que no colapsó

Deep SVDD tiene un modo de falla silencioso: si la red aprende a mapear *toda* entrada al centro de la esfera, el objetivo se minimiza perfectamente y el detector queda inútil — todos los puntos a la misma distancia, sin poder ordenar nada. Las capas sin sesgo lo hacen inalcanzable, pero conviene comprobarlo en vez de asumirlo.

In [ ]:
scores_svdd = resultados["deep_svdd"]["scores"]

print(f"desviación estándar del score: {scores_svdd.std():.4f}")
print(f"rango: [{scores_svdd.min():.4f}, {scores_svdd.max():.4f}]")
print("colapso de la hiperesfera:", "sí" if scores_svdd.std() < 1e-6 else "no")

## 2. Nivel cuenta: el autoencoder secuencial

Aquí cambia la representación de los datos, no solo el modelo.

**Por qué se agrupa por `nameDest` y no por `nameOrig`:** en PaySim el 99,85% de las cuentas de origen aparece una sola vez y ninguna llega a cinco transacciones — no hay historia que modelar del lado emisor. Las cuentas destino sí acumulan: 280.200 tienen cinco o más transacciones, cubriendo el 56,5% del dataset.

**Por qué se excluyen los comercios (`M`):** las 8.213 transacciones fraudulentas van todas a cuentas `C`. Dejar los comercios adentro le regalaría al modelo la regla "todo `M` es normal" y una métrica inflada que no mide detección de fraude.

In [ ]:
(seq_train, mask_train), (seq_test, mask_test), y_cuentas = get_sequence_data()

print(f"Train (cuentas limpias): {seq_train.shape}")
print(f"Test: {seq_test.shape} — cuentas con fraude: {int(y_cuentas.sum())} ({y_cuentas.mean():.2%})")
print(f"\nFeatures por paso: {FEATURE_NAMES}")
print(f"Largo real de las historias: mediana={np.median(mask_train.sum(axis=1)):.0f}, "
      f"máx={mask_train.sum(axis=1).max():.0f}")

El relleno va **al inicio** de la secuencia, para que la transacción más reciente quede siempre en la última posición y el estado final del encoder resuma la historia hasta el presente. La máscara excluye esas posiciones del error, de modo que una cuenta con historia corta no salga artificialmente rara solo por tener más relleno.

In [ ]:
modelo_seq = train_sequence_autoencoder(seq_train, mask_train)
errores_paso = sequence_step_errors(modelo_seq, seq_test, mask_test)
scores_seq = aggregate_step_errors(errores_paso, mask_test, how="mean")

metricas_seq = evaluate(pd.Series(y_cuentas), scores_seq)
print(f"PR-AUC (por cuenta): {metricas_seq['pr_auc']:.4f}  |  azar: {y_cuentas.mean():.4f}")
for k, (p, r) in metricas_seq["precision_recall_at_k"].items():
    print(f"  Precision@{k}: {p:.4f} | Recall@{k}: {r:.4f}")

### El resultado es pobre. ¿Es el método o son los datos?

PR-AUC 0.099 contra un azar de 0.087 es, en la práctica, no detectar nada. Antes de concluir algo hay que descartar las dos explicaciones que dependerían de decisiones mías y no del dataset.

**Primera: ¿el truncado deja el fraude fuera de la ventana?** Si al quedarme con las 20 transacciones más recientes estuviera descartando la transacción fraudulenta, el modelo nunca habría visto lo que debe detectar.

In [ ]:
cobertura = fraud_window_coverage(load_raw_data())

print(f"transacciones de fraude en cuentas elegibles: {cobertura['fraud_transactions']}")
print(f"  dentro de la ventana de 20: {cobertura['inside_window']} ({cobertura['coverage']:.1%})")
print(f"  posición mediana desde el final: {cobertura['median_position_from_end']:.0f}")

**Segunda: ¿la agregación diluye la señal?** Una cuenta puede tener una única transacción anómala entre veinte; promediar el error sobre toda la secuencia la aplasta. Se comparan cuatro formas de resumir los mismos errores por paso, sobre el mismo modelo ya entrenado — así la única variable es el score.

In [ ]:
for how in ["mean", "max", "top3", "last"]:
    alternativa = aggregate_step_errors(errores_paso, mask_test, how=how)
    m = evaluate(pd.Series(y_cuentas), alternativa)
    print(f"{how:5s} PR-AUC={m['pr_auc']:.4f}")

## 3. Resultados

Las dos tablas van separadas a propósito. La primera es comparable con el benchmark del Módulo 3; la segunda **no lo es**: evalúa cuentas, no transacciones, sobre un universo distinto y con una prevalencia distinta. Ponerlas en el mismo ranking sería un error de lectura, no un detalle de formato.

In [ ]:
fig = plot_deep_pr_curves(resultados, y_test, output_path=None)
plt.show()

In [ ]:
fig = plot_sequence_detector(
    {**metricas_seq, "scores": scores_seq}, pd.Series(y_cuentas), output_path=None
)
plt.show()

## 4. Conclusiones

**El objetivo importa más que la profundidad.** Deep SVDD (0.702) supera al autoencoder del Módulo 2 (0.581) con la misma arquitectura, los mismos datos y el mismo escalado: lo único que cambia es qué optimiza. Ese salto de +0.12 es mayor que el que separaba al autoencoder de su ablación lineal con PCA (+0.09), así que el margen estaba en la función objetivo, no en agregar capas. Y es barato: 9 s de ajuste y 4 ms de scoring.

**El VAE pierde contra el autoencoder simple** (0.446 vs 0.581) y es el más caro de los tres (39 s). El término KL empuja la latente hacia una normal estándar, lo que sobre features de colas muy pesadas termina normalizando justo la parte de la distribución que interesa. Ser la contraparte profunda del mejor detector clásico —GMM, un modelo de densidad— no le alcanzó para heredar su ventaja.

**El detector secuencial no encuentra señal, y el diagnóstico dice por qué no es culpa del método.** El 90% del fraude cae dentro de la ventana modelada, y las cuatro formas de agregar el error dan lo mismo (0.099–0.107 contra un azar de 0.087). Descartadas ambas explicaciones, queda la del dataset: PaySim inyecta el fraude con una regla fija y elige la cuenta destino sin modelar comportamiento de mula, así que las historias de cuenta no contienen el patrón que el modelo busca. Es una limitación del simulador, no del enfoque — sobre transacciones reales de AML esta es la familia que más aportaría, y la arquitectura queda lista para ese dato.

**Un resultado negativo bien diagnosticado vale más que uno bueno sin explicar.** El VAE además falló con `NaN` en la primera corrida sobre datos reales y solo funcionó tras diagnosticar la escala de las features. Un modelo probado únicamente con datos sintéticos bien portados no está probado.

Las métricas quedan persistidas en `data/processed/metrics.duckdb`: nivel transacción en `benchmark_metrics`, nivel cuenta en `sequence_metrics` — tablas separadas para que un `ORDER BY` descuidado no termine comparando dos problemas distintos.